# Sequence to sequence

Two RNNs pretending to be a translator. The bottleneck they hit is the reason attention exists.

## Problem Definition

Classification maps a variable-length sequence to a single label.
Translation maps a variable-length sequence to another variable length sequence.

The seq2seq architecture:
Two RNNs. One reads the source sentence and produces a fixed size context vector, the other reads that vector and generates the target sentence token by token.

## Basic Concept

### Encoder

An RNN that reads the source sentence. It's final state is the context vector ---- a fixed-size summery of the entire input.

### Decoder

Another RNN intialized from the context vector. At each step it takes the previously generated token as input and produces a distribution over the target vocabulary.

### Teacher forcing

During training, the decoder's input at step t is the groupd-truth token at poisition t-1, not decoder's own previous prediction.

### The bottle neck

The source must be squeezed into that one context vector. Long sentences lose detail, rare words get blurred, reordering has to be memeroized, not computed.

### Attention

Attention fixes this by letting the decoder look at every encoder hidden state, not just the last one.

# Build your Own

## An Encoder

In [1]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, src_vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(src_vocab_size, embed_dim, padding_idx=0)
        # Multi-layer Gated Recurrent Unit(GRU) RNN
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        e = self.embed(src)
        # outputs: (batch_size, seq_len, hidden_dim)
        # hidden: (1, batch_size, hidden_dim)
        outputs, hidden = self.gru(e)
        return outputs, hidden

## A decoder

In [3]:
class Decoder(nn.Module):
    def __init__(self, tgt_vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(tgt_vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, tgt_vocab_size)

    def forward(self, token, hidden):
        e = self.embed(token)
        out, hidden = self.gru(e, hidden)
        logits = self.fc(out)
        return logits, hidden

## Training Loop with teacher forcing

In [ ]:
def train_batch(encoder, decoder, src, tgt, bos_id, optimizer, teacher_forcing_ratio=0.9):
    optimizer.zero_grad()

    # Squeeze source sentence into a context vector
    _, hidden = encoder(src)
    batch_size, tgt_len = tgt.shape
    input_token = torch.full((batch_size, 1), bos_id, dtype=torch.long)
    loss = 0.0
    loss_fn = nn.CrossEntropyLoss(ignore_index=0)

    for t in range(tgt_len):
        # Recursively generate target sentence
        logits, hidden = decoder(input_token, hidden)
        step_loss = loss_fn(logits.squeeze(1), tgt[:, t])
        loss += step_loss
        use_teacher = torch.rand(1).item() < teacher_forcing_ratio
        if use_teacher:
            input_token = tgt[:, t].unsqueeze(1)
        else:
            input_token = logits.argmax(dim=-1)

    loss.backward()
    optimizer.step()

    return loss.item() / tgt_len

## Inference Loop

In [ ]:
@torch.no_grad()
def greedy_decode(encoder, decoder, src, bos_id, eos_id, max_len=50):
    _, hidden = encoder(src)
    batch_size = src.shape[0]
    input_token = torch.full((batch_size, 1), bos_id, dtype=torch.long)
    output_ids = []

    for _ in range(max_len):
        logits, hidden = decoder(input_token, hidden)
        next_token = logits.argmax(dim=-1)
        # Append one token for each batch
        output_ids.append(next_token)
        input_token = next_token
        if (next_token == eos_id).all():
            break

    # Concatenate all tokens
    return torch.cat(output_ids, dim=1)

## From Existing

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

# facebook/bart-base is English denoising — it will not translate.
# Helsinki MarianMT is trained specifically for EN→FR.
model_name = "Helsinki-NLP/opus-mt-en-fr"
tok = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

src = tok("Hello, how are you?", return_tensors="pt")
out = model.generate(**src, max_new_tokens=50, num_beams=4)
print(tok.decode(out[0], skip_special_tokens=True))
